# DP defense for Graph RAG - self-built backend 

No LightRAG. We build the medical knowledge graph ourselves with one LLM call
per document, retrieve by cosine similarity (which gives the real scores the DP
mechanism needs), then run the attack, the DP defense, and the [P7] baselines.

**Why this design:** LightRAG's async pipeline caused every failure (GPU load
crashes, context overflow, a dedup registry). Here every LLM call is one plain
HTTP request, one document at a time, resumable, with nothing to crash under
concurrency. A failed doc is retried alone.

**Pipeline (same for every condition):**
`retrieve(query, store)` -> `defense.apply(retrieved, session)` -> `answer(...)`

Run A1-A3 once per runtime. B builds the graph (leave it running). C runs the attack.

**Runtime > Change runtime type > T4 GPU.**

## A1. System deps + code path [every runtime]

In [ ]:
!apt-get -qq update && apt-get -qq install -y zstd pciutils lshw >/dev/null 2>&1
from google.colab import drive
drive.mount('/content/drive')
import sys
ROOT = '/content/drive/MyDrive/graphrag_defense'
sys.path.insert(0, ROOT)
!lspci | grep -i nvidia

## A2. Start Ollama on the GPU, models pinned [every runtime]

Both models pinned at 4096 context (the setting that loads cleanly on a T4).
`api/ps` must list both with size_vram > 0.

In [ ]:
%%bash
pkill -9 -f ollama 2>/dev/null; sleep 3
curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -1
nohup env OLLAMA_KEEP_ALIVE=-1 ollama serve > /tmp/ollama.log 2>&1 &
sleep 8
ollama pull qwen2.5:7b >/dev/null 2>&1
ollama pull nomic-embed-text >/dev/null 2>&1
curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:7b","prompt":"hi","options":{"num_ctx":4096},"keep_alive":-1,"stream":false}' >/dev/null
curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text","prompt":"hi","keep_alive":-1}' >/dev/null
echo "--- loaded (want both, size_vram>0) ---"; curl -s http://localhost:11434/api/ps

## A3. Imports + data [every runtime]

In [ ]:
!pip install -q requests networkx numpy datasets pandas rouge-score >/dev/null 2>&1
from grd import config as C, data, attack, metrics, kg_build as kb, retriever as rt, run_b
from grd.defenses import NoDefense, DPRetrieval, SystemPromptRule, SimilarityThreshold
C.ensure_dirs()
docs, records, idx = data.load_subset(n_docs=100)
qa = data.build_benign_qa(records)
print(len(docs), 'docs, unique:', len(set(docs)))

## A4. Smoke-test the LLM extraction on ONE doc

Confirms the model returns parseable graph JSON before you index 100. Expect a
handful of entities and relations, in a few seconds.

In [ ]:
obj = kb.extract_one(docs[0])
print('entities:', len(obj['entities']), '| relations:', len(obj['relations']))
print(obj['entities'][:3])
print(obj['relations'][:2])

## B. Build the graph (resumable, leave running)

One LLM call per doc + embeddings, saved after each doc. Expect a few seconds to
~30s per doc on the GPU. 100 docs is roughly 10-40 min. Rerun this cell after any
reset and it continues from `done.json`.

In [ ]:
STORE_DIR = f'{ROOT}/kg/hcm100'
store = kb.KGStore(STORE_DIR)
print('resuming from', len(store.done), 'docs;', store.stats(), 'nodes/edges')
kb.build_graph(docs, store)
print('\nFINAL GRAPH:', store.stats())

### B1. Graph stats for the paper's Setup section

In [ ]:
N_ENT, N_REL, ALL_ENT, ALL_REL = run_b.graph_sets(store)
print(f'entities={N_ENT}  relationships={N_REL}  docs={len(store.done)}')

## C. Reproduce the attack

### C1. Retrieval check - confirm scored candidates come back
DP selection needs real scores in [0,1]. This must show entities with nonzero,
varied scores.

In [ ]:
ret = rt.retrieve('I want information about diabetes', store, top_n=20)
print('entities:', len(ret.entities), '| relations:', len(ret.relations))
for e in ret.entities[:5]:
    print(f'  {e.name:30s} score={e.score:.3f}')

### C2. Smoke test - C1 vs C3, 20 targeted queries (no defense)
Checkpoint: C3 entity/relationship leakage clearly above C1.
Paste the exact [P7] command/prefix strings into grd/attack.py before the full run.

In [ ]:
import grd.kg_build as kb
_orig = kb.ollama_generate
def gen_capped(prompt, system='', model=None, host=None, num_ctx=4096, timeout=600):
    import requests
    from grd import config as C
    r = requests.post(f'{C.OLLAMA_HOST}/api/generate', json={
        'model': C.LLM_MODEL, 'prompt': prompt, 'system': system,
        'options': {'temperature': 0.0, 'num_ctx': num_ctx, 'num_predict': 512},
        'keep_alive': -1, 'stream': False}, timeout=timeout)
    r.raise_for_status()
    return r.json()['response']
kb.ollama_generate = gen_capped
print('answer generation: timeout 600s, output capped at 512 tokens')

In [ ]:
from grd import config as C
C.DP_CANDIDATE_POOL = 30   # top-30 entities + top-30 relations per query
print('pool =', C.DP_CANDIDATE_POOL)

In [ ]:
%%bash
pkill -9 -f ollama 2>/dev/null; sleep 3
nohup env OLLAMA_KEEP_ALIVE=-1 ollama serve > /tmp/ollama.log 2>&1 &
sleep 8
curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:7b","prompt":"hi","options":{"num_ctx":4096},"keep_alive":-1,"stream":false}' >/dev/null
curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text","prompt":"hi","keep_alive":-1}' >/dev/null
echo "--- both models should be listed ---"
curl -s http://localhost:11434/api/ps

In [ ]:
# speed patches
import grd.kg_build as kb
from grd import config as C
def gen_capped(prompt, system='', model=None, host=None, num_ctx=4096, timeout=600):
    import requests
    r = requests.post(f'{C.OLLAMA_HOST}/api/generate', json={
        'model': C.LLM_MODEL, 'prompt': prompt, 'system': system,
        'options': {'temperature':0.0,'num_ctx':num_ctx,'num_predict':512},
        'keep_alive':-1,'stream':False}, timeout=timeout)
    r.raise_for_status(); return r.json()['response']
kb.ollama_generate = gen_capped
C.DP_CANDIDATE_POOL = 30

# clean the failed smoke logs and rerun
import os
for f in ['smoke_targeted_C3_none','smoke_targeted_C1_none']:
    p = f'{C.LOG_DIR}/{f}.jsonl'
    if os.path.exists(p): os.remove(p)
for cmd in ['C3','C1']:
    qs = attack.build_queries('targeted', cmd)[:20]
    run_b.run(store, qs, f'smoke_targeted_{cmd}_none', defense=NoDefense())
metrics.summarize_runs(['smoke_targeted_C3_none','smoke_targeted_C1_none'], N_ENT, N_REL, C.LOG_DIR)

## D. The DP defense (once the checkpoint passes)

`DPRetrieval` now runs over the real scored pool: report-noisy-max top-k for
entities and relationships (Delta=1 on the [0,1] scores), per-stage epsilon,
optional numeric perturbation, per-user budget via the session.

In [ ]:
%%bash
pkill -9 -f ollama 2>/dev/null; sleep 3
nohup env OLLAMA_KEEP_ALIVE=-1 OLLAMA_MAX_QUEUE=1 ollama serve > /tmp/ollama.log 2>&1 &
sleep 8
curl -s http://localhost:11434/api/generate -d '{"model":"qwen2.5:7b","prompt":"hi","options":{"num_ctx":4096},"keep_alive":-1,"stream":false}' >/dev/null
curl -s http://localhost:11434/api/embeddings -d '{"model":"nomic-embed-text","prompt":"hi","keep_alive":-1}' >/dev/null
curl -s http://localhost:11434/api/ps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys; sys.path.insert(0, '/content/drive/MyDrive/graphrag_defense')

from grd import config as C, data, attack, metrics, kg_build as kb, retriever as rt, run_b
from grd.defenses import NoDefense, DPRetrieval, SystemPromptRule, SimilarityThreshold
C.ensure_dirs()
ROOT = '/content/drive/MyDrive/graphrag_defense'

# reload the graph (no rebuild)
store = kb.KGStore(f'{ROOT}/kg/hcm100')
N_ENT, N_REL, ALL_ENT, ALL_REL = run_b.graph_sets(store)
print('graph:', N_ENT, N_REL)

# re-apply the two speed patches (lost on reset)
def gen_capped(prompt, system='', model=None, host=None, num_ctx=4096, timeout=600):
    import requests
    r = requests.post(f'{C.OLLAMA_HOST}/api/generate', json={
        'model': C.LLM_MODEL, 'prompt': prompt, 'system': system,
        'options': {'temperature':0.0,'num_ctx':num_ctx,'num_predict':512},
        'keep_alive':-1,'stream':False}, timeout=timeout)
    r.raise_for_status(); return r.json()['response']
kb.ollama_generate = gen_capped
C.DP_CANDIDATE_POOL = 30
print('ready: graph loaded, patches applied')

In [ ]:
NQ = 40
runs = ['attack_targeted_C3_none']
run_b.run(store, attack.build_queries('targeted','C3')[:NQ], 'attack_targeted_C3_none', defense=NoDefense())
for eps in [0.5, 1.0, 2.0]:
    name = f'attack_targeted_C3_dp{eps}'
    run_b.run(store, attack.build_queries('targeted','C3')[:NQ], name, defense=DPRetrieval(eps_ent=eps, eps_rel=eps, k=10))
    runs.append(name)
metrics.summarize_runs(runs, N_ENT, N_REL, C.LOG_DIR)

In [ ]:
import grd.kg_build as kb, time, requests
from grd import config as C
def gen_retry(prompt, system='', model=None, host=None, num_ctx=4096, timeout=600):
    last = None
    for attempt in range(3):
        try:
            r = requests.post(f'{C.OLLAMA_HOST}/api/generate', json={
                'model': C.LLM_MODEL, 'prompt': prompt, 'system': system,
                'options': {'temperature':0.0,'num_ctx':num_ctx,'num_predict':512},
                'keep_alive':-1,'stream':False}, timeout=timeout)
            r.raise_for_status(); return r.json()['response']
        except Exception as e:
            last = e; time.sleep(5)
    raise last
def emb_retry(text, model=None, host=None, timeout=120):
    for attempt in range(3):
        try:
            r = requests.post(f'{C.OLLAMA_HOST}/api/embeddings', json={
                'model': C.EMB_MODEL, 'prompt': text, 'keep_alive':-1}, timeout=timeout)
            r.raise_for_status(); return r.json()['embedding']
        except Exception:
            time.sleep(5)
    raise RuntimeError('embed failed after retries')
kb.ollama_generate = gen_retry
kb.ollama_embed = emb_retry
import grd.retriever as rt; rt.ollama_embed = emb_retry
print('retry wrappers installed')

In [ ]:
import os
for eps in [0.5,1.0,2.0]:
    p = f'{C.LOG_DIR}/attack_targeted_C3_dp{eps}.jsonl'
    if os.path.exists(p): os.remove(p)
runs = ['attack_targeted_C3_none']
for eps in [0.5,1.0,2.0]:
    name = f'attack_targeted_C3_dp{eps}'
    run_b.run(store, attack.build_queries('targeted','C3')[:40], name, defense=DPRetrieval(eps_ent=eps, eps_rel=eps, k=10))
    runs.append(name)
metrics.summarize_runs(runs, N_ENT, N_REL, C.LOG_DIR)

In [ ]:
NQ = 40   # quick pass; raise to 250 for the final numbers
runs = ['attack_targeted_C3_none']
run_b.run(store, attack.build_queries('targeted','C3')[:NQ], 'attack_targeted_C3_none', defense=NoDefense())
for eps in [0.5, 1.0, 2.0]:
    name = f'attack_targeted_C3_dp{eps}'
    dp = DPRetrieval(eps_ent=eps, eps_rel=eps, k=10)
    run_b.run(store, attack.build_queries('targeted','C3')[:NQ], name, defense=dp)
    runs.append(name)
metrics.summarize_runs(runs, N_ENT, N_REL, C.LOG_DIR)

In [ ]:
# no-defense vs DP at three epsilon values, targeted C3
runs = ['attack_targeted_C3_none']
run_b.run(store, attack.build_queries('targeted','C3'), 'attack_targeted_C3_none', defense=NoDefense())
for eps in [0.5, 1.0, 2.0]:
    name = f'attack_targeted_C3_dp{eps}'
    dp = DPRetrieval(eps_ent=eps, eps_rel=eps, k=C.DP_TOP_K)
    run_b.run(store, attack.build_queries('targeted','C3'), name, defense=dp)
    runs.append(name)
metrics.summarize_runs(runs, N_ENT, N_REL, C.LOG_DIR)

**Next:** the three [P7] baselines (SystemPromptRule, SimilarityThreshold,
Summarization), the epsilon sweep + privacy-utility frontier, and the
repeated-query aggregation experiment. All reuse `run_b.run(...)`. Say the word
and I add the Step-D notebook once the smoke test passes.